In [6]:
#!/usr/bin/env python3
"""
Interactive Level 2 fMRI Group Analysis Viewer for Jupyter Notebooks
Real-time visualization with ipywidgets controls
"""

import os
import glob
import numpy as np
import nibabel as nib
from scipy.io import loadmat
from scipy import stats
import matplotlib.pyplot as plt
from nilearn import plotting, datasets
from nilearn.image import threshold_img
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

class InteractiveLevel2Viewer:
    """Interactive Level 2 fMRI viewer with Jupyter widgets"""
    
    def __init__(self, level2_path):
        self.level2_path = level2_path
        self.contrasts = []
        self.current_img = None
        self.current_contrast = None
        
        # Initialize widgets
        self.setup_widgets()
        
        # Load contrasts
        self.load_contrasts()
        
        # Update contrast dropdown
        self.update_contrast_dropdown()
    
    def setup_widgets(self):
        """Initialize all interactive widgets"""
        
        # Contrast selection
        self.contrast_dropdown = widgets.Dropdown(
            options=[],
            description='Contrast:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )
        
        # Correction method
        self.correction_dropdown = widgets.Dropdown(
            options=[
                ('None (uncorrected)', 'none'),
                ('FDR correction', 'fdr'),
                ('Bonferroni correction', 'bonferroni')
            ],
            value='fdr',
            description='Correction:',
            style={'description_width': 'initial'}
        )
        
        # Alpha level
        self.alpha_slider = widgets.FloatSlider(
            value=0.05,
            min=0.001,
            max=0.1,
            step=0.001,
            description='Alpha (α):',
            readout_format='.3f',
            style={'description_width': 'initial'}
        )
        
        # Uncorrected threshold (only used when correction='none')
        self.threshold_slider = widgets.FloatSlider(
            value=2.0,
            min=1.0,
            max=5.0,
            step=0.1,
            description='Threshold:',
            readout_format='.1f',
            style={'description_width': 'initial'}
        )
        
        # Image type
        self.image_type_dropdown = widgets.Dropdown(
            options=[
                ('T-statistics', 'spmT'),
                ('Contrast estimates', 'contrast')
            ],
            value='spmT',
            description='Image type:',
            style={'description_width': 'initial'}
        )
        
        # Display mode
        self.display_mode_dropdown = widgets.Dropdown(
            options=[
                ('Glass brain', 'glass_brain'),
                ('Orthogonal slices', 'ortho'),
                ('Mosaic', 'mosaic'),
                ('Sagittal (x)', 'x'),
                ('Coronal (y)', 'y'),
                ('Axial (z)', 'z')
            ],
            value='glass_brain',
            description='Display:',
            style={'description_width': 'initial'}
        )
        
        # Action buttons
        self.update_button = widgets.Button(
            description='Update View',
            button_style='primary',
            icon='refresh'
        )
        
        self.compare_button = widgets.Button(
            description='Compare Corrections',
            button_style='info',
            icon='bar-chart'
        )
        
        self.summary_button = widgets.Button(
            description='Threshold Summary',
            button_style='success',
            icon='table'
        )
        
        self.export_button = widgets.Button(
            description='Export Maps',
            button_style='warning',
            icon='download'
        )
        
        # Output areas
        self.plot_output = widgets.Output()
        self.stats_output = widgets.Output()
        self.info_output = widgets.Output()
        
        # Set up event handlers
        self.setup_event_handlers()
    
    def setup_event_handlers(self):
        """Set up widget event handlers"""
        self.contrast_dropdown.observe(self.on_contrast_change, names='value')
        self.correction_dropdown.observe(self.on_correction_change, names='value')
        self.alpha_slider.observe(self.on_parameter_change, names='value')
        self.threshold_slider.observe(self.on_parameter_change, names='value')
        self.image_type_dropdown.observe(self.on_parameter_change, names='value')
        self.display_mode_dropdown.observe(self.on_parameter_change, names='value')
        
        self.update_button.on_click(self.update_view)
        self.compare_button.on_click(self.compare_corrections)
        self.summary_button.on_click(self.show_summary)
        self.export_button.on_click(self.export_maps)
    
    def load_contrasts(self):
        """Load contrast information from directories"""
        if not os.path.exists(self.level2_path):
            with self.info_output:
                clear_output()
                print(f"❌ Level 2 directory not found: {self.level2_path}")
            return
        
        # Find contrast directories
        contrast_dirs = glob.glob(os.path.join(self.level2_path, "*"))
        contrast_dirs = [d for d in contrast_dirs if os.path.isdir(d)]
        contrast_dirs.sort()
        
        self.contrasts = []
        
        with self.info_output:
            clear_output()
            print(f"🔍 Loading contrasts from: {os.path.basename(self.level2_path)}")
            print(f"Found {len(contrast_dirs)} directories")
            print()
        
        for contrast_dir in contrast_dirs:
            contrast_name = os.path.basename(contrast_dir)
            
            # Look for files
            spm_mat_path = os.path.join(contrast_dir, 'SPM.mat')
            contrast_file = os.path.join(contrast_dir, 'con_0001.nii')
            spmT_file = os.path.join(contrast_dir, 'spmT_0001.nii')
            
            # Alternative naming
            if not os.path.exists(contrast_file):
                contrast_file = os.path.join(contrast_dir, 'contrast.nii')
            if not os.path.exists(spmT_file):
                spmT_file = os.path.join(contrast_dir, 'spmT.nii')
            
            files_found = {
                'spm_mat': spm_mat_path if os.path.exists(spm_mat_path) else None,
                'contrast': contrast_file if os.path.exists(contrast_file) else None,
                'spmT': spmT_file if os.path.exists(spmT_file) else None
            }
            
            # Extract info from SPM.mat
            display_name = contrast_name
            df_error = None
            n_subjects = None
            
            if files_found['spm_mat']:
                try:
                    spm_data = loadmat(files_found['spm_mat'], struct_as_record=False, squeeze_me=True)
                    spm = spm_data['SPM']
                    
                    if hasattr(spm, 'xCon') and spm.xCon.size > 0:
                        xCon = spm.xCon
                        if not isinstance(xCon, np.ndarray):
                            xCon = [xCon]
                        if len(xCon) > 0:
                            display_name = xCon[0].name if hasattr(xCon[0], 'name') else contrast_name
                    
                    if hasattr(spm, 'xX') and hasattr(spm.xX, 'erdf'):
                        df_error = spm.xX.erdf
                    
                    # Try to get sample size
                    if hasattr(spm, 'xY') and hasattr(spm.xY, 'P'):
                        if hasattr(spm.xY.P, 'shape'):
                            n_subjects = spm.xY.P.shape[0] if len(spm.xY.P.shape) > 0 else None
                        elif hasattr(spm.xY.P, '__len__'):
                            n_subjects = len(spm.xY.P)
                
                except Exception as e:
                    with self.info_output:
                        print(f"⚠️ Could not read SPM.mat in {contrast_name}: {e}")
            
            contrast_info = {
                'index': len(self.contrasts),
                'folder_name': contrast_name,
                'display_name': display_name,
                'path': contrast_dir,
                'files': files_found,
                'df_error': df_error,
                'n_subjects': n_subjects
            }
            
            self.contrasts.append(contrast_info)
            
            # Status update
            with self.info_output:
                status = "✅" if all(files_found.values()) else "⚠️"
                missing = [k for k, v in files_found.items() if v is None]
                missing_str = f" (missing: {', '.join(missing)})" if missing else ""
                print(f"  {status} {display_name}{missing_str}")
                if n_subjects:
                    print(f"      N={n_subjects} subjects")
        
        with self.info_output:
            print(f"\n✅ Loaded {len(self.contrasts)} contrasts")
    
    def update_contrast_dropdown(self):
        """Update contrast dropdown options"""
        options = []
        for i, contrast in enumerate(self.contrasts):
            label = f"{i+1}. {contrast['display_name']}"
            if contrast['n_subjects']:
                label += f" (N={contrast['n_subjects']})"
            options.append((label, i))
        
        self.contrast_dropdown.options = options
        if options:
            self.contrast_dropdown.value = 0
    
    def load_statistical_image(self, contrast_idx, image_type='spmT'):
        """Load statistical image for a contrast"""
        if contrast_idx >= len(self.contrasts):
            return None, None
        
        contrast = self.contrasts[contrast_idx]
        
        # Determine file to load
        if image_type == 'spmT' and contrast['files']['spmT']:
            img_path = contrast['files']['spmT']
        elif image_type == 'contrast' and contrast['files']['contrast']:
            img_path = contrast['files']['contrast']
        else:
            # Fallback
            for file_type in ['spmT', 'contrast']:
                if contrast['files'][file_type]:
                    img_path = contrast['files'][file_type]
                    break
            else:
                return None, None
        
        try:
            img = nib.load(img_path)
            return img, contrast
        except Exception as e:
            with self.stats_output:
                print(f"❌ Error loading {img_path}: {e}")
            return None, None
    
    def apply_fdr_correction(self, stat_img, alpha=0.05, df_error=None):
        """Apply FDR correction"""
        data = stat_img.get_fdata()
        mask = data != 0
        stat_values = data[mask]
        
        if len(stat_values) == 0:
            return None, None, 0
        
        # Convert to p-values
        if df_error and df_error > 0:
            p_values = 2 * (1 - stats.t.cdf(np.abs(stat_values), df_error))
        else:
            p_values = 2 * (1 - stats.norm.cdf(np.abs(stat_values)))
        
        # Benjamini-Hochberg procedure
        sorted_p = np.sort(p_values)
        m = len(p_values)
        fdr_line = alpha * np.arange(1, m + 1) / m
        significant = sorted_p <= fdr_line
        
        if np.any(significant):
            fdr_p_threshold = sorted_p[np.where(significant)[0][-1]]
            
            if df_error and df_error > 0:
                fdr_t_threshold = stats.t.ppf(1 - fdr_p_threshold/2, df_error)
            else:
                fdr_t_threshold = stats.norm.ppf(1 - fdr_p_threshold/2)
            
            thresholded_img = threshold_img(stat_img, threshold=fdr_t_threshold)
            n_significant = np.sum(significant)
            
            return thresholded_img, fdr_t_threshold, n_significant
        else:
            return None, None, 0
    
    def apply_bonferroni_correction(self, stat_img, alpha=0.05, df_error=None):
        """Apply Bonferroni correction"""
        data = stat_img.get_fdata()
        mask = data != 0
        n_voxels = np.sum(mask)
        
        bonferroni_alpha = alpha / n_voxels
        
        if df_error and df_error > 0:
            bonferroni_t = stats.t.ppf(1 - bonferroni_alpha/2, df_error)
        else:
            bonferroni_t = stats.norm.ppf(1 - bonferroni_alpha/2)
        
        thresholded_img = threshold_img(stat_img, threshold=bonferroni_t)
        return thresholded_img, bonferroni_t
    
    def create_plot(self, contrast_idx, correction='fdr', threshold=2.0, alpha=0.05, 
                   image_type='spmT', display_mode='glass_brain'):
        """Create the main plot"""
        
        img, contrast = self.load_statistical_image(contrast_idx, image_type)
        if img is None:
            with self.plot_output:
                clear_output()
                print("❌ Could not load image")
            return
        
        self.current_img = img
        self.current_contrast = contrast
        
        df_error = contrast['df_error']
        
        # Apply thresholding
        if correction == 'none':
            try:
                thresholded_img = threshold_img(img, threshold=threshold)
                threshold_desc = f"t > {threshold}"
                n_significant = np.sum(np.abs(thresholded_img.get_fdata()) > 0)
            except:
                thresholded_img = img
                threshold_desc = "unthresholded"
                n_significant = np.sum(img.get_fdata() != 0)
        
        elif correction == 'fdr':
            result = self.apply_fdr_correction(img, alpha, df_error)
            thresholded_img, fdr_threshold, n_significant = result
            if thresholded_img is not None:
                threshold_desc = f"FDR q < {alpha} (t > {fdr_threshold:.3f})"
            else:
                with self.plot_output:
                    clear_output()
                    print(f"⚠️ No voxels survive FDR correction at q < {alpha}")
                return
        
        elif correction == 'bonferroni':
            thresholded_img, bonf_threshold = self.apply_bonferroni_correction(img, alpha, df_error)
            threshold_desc = f"Bonferroni p < {alpha} (t > {bonf_threshold:.3f})"
            n_significant = np.sum(np.abs(thresholded_img.get_fdata()) > 0)
        
        # Create title
        title = f"Level 2: {contrast['display_name']}\n{threshold_desc}"
        if contrast['n_subjects']:
            title += f" (N={contrast['n_subjects']})"
        
        # Create plot
        with self.plot_output:
            clear_output(wait=True)
            
            if display_mode == 'glass_brain':
                fig = plt.figure(figsize=(16, 6))
                plotting.plot_glass_brain(
                    thresholded_img,
                    colorbar=True,
                    title=title,
                    plot_abs=False,
                    display_mode='lyrz',
                    figure=fig
                )
            else:
                # Load anatomical template
                anat_img = datasets.load_mni152_template(resolution=2)
                fig = plt.figure(figsize=(15, 5))
                
                plotting.plot_stat_map(
                    thresholded_img,
                    bg_img=anat_img,
                    display_mode=display_mode,
                    colorbar=True,
                    title=title,
                    figure=fig
                )
            
            plt.tight_layout()
            plt.show()
        
        # Update stats
        with self.stats_output:
            clear_output()
            if n_significant > 0:
                data = thresholded_img.get_fdata()
                max_val = np.max(data)
                min_val = np.min(data)
                
                print("📊 ACTIVATION STATISTICS")
                print("=" * 30)
                print(f"Significant voxels: {n_significant:,}")
                print(f"Peak T-value: {max_val:.3f}")
                print(f"Min T-value: {min_val:.3f}")
                if contrast['n_subjects']:
                    print(f"Sample size: N={contrast['n_subjects']}")
                if df_error:
                    print(f"Degrees of freedom: {df_error}")
                
                # Calculate percentage of brain
                total_voxels = np.sum(img.get_fdata() != 0)
                percentage = (n_significant / total_voxels) * 100 if total_voxels > 0 else 0
                print(f"Brain coverage: {percentage:.2f}%")
                
            else:
                print("📊 No significant voxels found with current threshold")
    
    def on_contrast_change(self, change):
        """Handle contrast selection change"""
        self.update_view(None)
    
    def on_correction_change(self, change):
        """Handle correction method change"""
        # Show/hide threshold slider based on correction method
        if change['new'] == 'none':
            self.threshold_slider.layout.display = 'flex'
        else:
            self.threshold_slider.layout.display = 'none'
        
        self.update_view(None)
    
    def on_parameter_change(self, change):
        """Handle parameter changes"""
        # Auto-update if desired (can be made optional)
        pass
    
    def update_view(self, button):
        """Update the main view"""
        if not self.contrasts:
            return
        
        contrast_idx = self.contrast_dropdown.value
        correction = self.correction_dropdown.value
        alpha = self.alpha_slider.value
        threshold = self.threshold_slider.value
        image_type = self.image_type_dropdown.value
        display_mode = self.display_mode_dropdown.value
        
        self.create_plot(contrast_idx, correction, threshold, alpha, image_type, display_mode)
    
    def compare_corrections(self, button):
        """Compare different correction methods"""
        if not self.contrasts:
            return
        
        contrast_idx = self.contrast_dropdown.value
        alpha = self.alpha_slider.value
        image_type = self.image_type_dropdown.value
        
        img, contrast = self.load_statistical_image(contrast_idx, image_type)
        if img is None:
            return
        
        with self.plot_output:
            clear_output(wait=True)
            
            fig, axes = plt.subplots(2, 2, figsize=(20, 12))
            
            # Uncorrected
            try:
                uncorr_img = threshold_img(img, threshold=2.0)
                plotting.plot_glass_brain(uncorr_img, axes=axes[0,0], colorbar=True,
                                        title="Uncorrected (t > 2.0)", display_mode='z')
            except:
                axes[0,0].text(0.5, 0.5, 'Error', ha='center', va='center', transform=axes[0,0].transAxes)
            
            # FDR
            result = self.apply_fdr_correction(img, alpha, contrast['df_error'])
            if result[0] is not None:
                fdr_img, fdr_t, _ = result
                plotting.plot_glass_brain(fdr_img, axes=axes[0,1], colorbar=True,
                                        title=f"FDR (q < {alpha})", display_mode='z')
            else:
                axes[0,1].text(0.5, 0.5, 'No FDR\nsurvivors', ha='center', va='center',
                              transform=axes[0,1].transAxes, fontsize=14)
                axes[0,1].set_title(f"FDR (q < {alpha})")
            
            # Bonferroni
            bonf_img, bonf_t = self.apply_bonferroni_correction(img, alpha, contrast['df_error'])
            plotting.plot_glass_brain(bonf_img, axes=axes[1,0], colorbar=True,
                                    title=f"Bonferroni (p < {alpha})", display_mode='z')
            
            # Liberal
            try:
                liberal_img = threshold_img(img, threshold=1.96)
                plotting.plot_glass_brain(liberal_img, axes=axes[1,1], colorbar=True,
                                        title="Liberal (t > 1.96)", display_mode='z')
            except:
                axes[1,1].text(0.5, 0.5, 'Error', ha='center', va='center', transform=axes[1,1].transAxes)
            
            plt.suptitle(f"Threshold Comparison - {contrast['display_name']}", fontsize=16)
            plt.tight_layout()
            plt.show()
    
    def show_summary(self, button):
        """Show detailed threshold summary"""
        if not self.contrasts:
            return
        
        contrast_idx = self.contrast_dropdown.value
        alpha = self.alpha_slider.value
        image_type = self.image_type_dropdown.value
        
        img, contrast = self.load_statistical_image(contrast_idx, image_type)
        if img is None:
            return
        
        with self.stats_output:
            clear_output()
            
            print("📋 COMPREHENSIVE THRESHOLD SUMMARY")
            print("=" * 50)
            print(f"Contrast: {contrast['display_name']}")
            print(f"Image type: {image_type}")
            if contrast['n_subjects']:
                print(f"Sample size: N={contrast['n_subjects']}")
            if contrast['df_error']:
                print(f"Degrees of freedom: {contrast['df_error']}")
            print()
            
            data = img.get_fdata()
            mask = data != 0
            n_total = np.sum(mask)
            
            print(f"📊 Total brain voxels: {n_total:,}")
            print()
            
            # Uncorrected thresholds
            print("🎯 UNCORRECTED THRESHOLDS:")
            thresholds = [1.96, 2.0, 2.58, 3.0, 3.29, 4.0]
            for t_thresh in thresholds:
                n_sig = np.sum(np.abs(data[mask]) > t_thresh)
                pct = (n_sig / n_total) * 100 if n_total > 0 else 0
                
                if contrast['df_error']:
                    p_val = 2 * (1 - stats.t.cdf(t_thresh, contrast['df_error']))
                else:
                    p_val = 2 * (1 - stats.norm.cdf(t_thresh))
                
                print(f"   t > {t_thresh}: {n_sig:,} voxels ({pct:.2f}%) - p < {p_val:.4f}")
            print()
            
            # FDR correction
            print("🧮 FDR CORRECTION:")
            result = self.apply_fdr_correction(img, alpha, contrast['df_error'])
            _, fdr_threshold, n_fdr = result
            if fdr_threshold is not None:
                pct_fdr = (n_fdr / n_total) * 100 if n_total > 0 else 0
                print(f"   FDR q < {alpha}: t > {fdr_threshold:.3f}")
                print(f"   Significant: {n_fdr:,} voxels ({pct_fdr:.2f}%)")
            else:
                print(f"   No voxels survive FDR q < {alpha}")
            print()
            
            # Bonferroni correction
            print("🔒 BONFERRONI CORRECTION:")
            bonf_img, bonf_threshold = self.apply_bonferroni_correction(img, alpha, contrast['df_error'])
            bonf_data = bonf_img.get_fdata()
            n_bonf = np.sum(np.abs(bonf_data) > 0)
            pct_bonf = (n_bonf / n_total) * 100 if n_total > 0 else 0
            print(f"   Bonferroni p < {alpha}: t > {bonf_threshold:.3f}")
            print(f"   Significant: {n_bonf:,} voxels ({pct_bonf:.2f}%)")
    
    def export_maps(self, button):
        """Export thresholded maps"""
        if not self.contrasts or self.current_img is None:
            with self.stats_output:
                print("❌ No image loaded to export")
            return
        
        contrast_idx = self.contrast_dropdown.value
        alpha = self.alpha_slider.value
        
        output_dir = os.path.join(self.level2_path, "exported_maps")
        os.makedirs(output_dir, exist_ok=True)
        
        contrast = self.contrasts[contrast_idx]
        img = self.current_img
        
        with self.stats_output:
            print("💾 EXPORTING THRESHOLDED MAPS...")
            print(f"Output directory: {output_dir}")
            print()
        
        # Export FDR map
        result = self.apply_fdr_correction(img, alpha, contrast['df_error'])
        if result[0] is not None:
            fdr_img, _, _ = result
            fdr_path = os.path.join(output_dir, f"{contrast['folder_name']}_FDR_{alpha}.nii")
            nib.save(fdr_img, fdr_path)
            with self.stats_output:
                print(f"✅ Saved FDR map: {os.path.basename(fdr_path)}")
        
        # Export Bonferroni map
        bonf_img, _ = self.apply_bonferroni_correction(img, alpha, contrast['df_error'])
        bonf_path = os.path.join(output_dir, f"{contrast['folder_name']}_Bonferroni_{alpha}.nii")
        nib.save(bonf_img, bonf_path)
        
        with self.stats_output:
            print(f"✅ Saved Bonferroni map: {os.path.basename(bonf_path)}")
            print(f"\n📁 All maps saved to: {output_dir}")
    
    def create_interface(self):
        """Create the main interface"""
        
        # Title
        title_html = """
        <h2>🧠 Interactive Level 2 fMRI Group Analysis Viewer</h2>
        <p>Select contrast, correction method, and visualization options below.</p>
        """
        
        # Control panels
        controls_box1 = widgets.VBox([
            widgets.HTML("<h3>📊 Analysis Controls</h3>"),
            self.contrast_dropdown,
            self.image_type_dropdown,
            self.correction_dropdown,
            self.alpha_slider,
            self.threshold_slider
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px'))
        
        controls_box2 = widgets.VBox([
            widgets.HTML("<h3>🎨 Display Controls</h3>"),
            self.display_mode_dropdown,
            self.update_button,
            widgets.HTML("<br>"),
            self.compare_button,
            self.summary_button,
            self.export_button
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px'))
        
        controls_row = widgets.HBox([controls_box1, controls_box2])
        
        # Output panels
        plot_box = widgets.VBox([
            widgets.HTML("<h3>📈 Visualization</h3>"),
            self.plot_output
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px'))
        
        info_box = widgets.VBox([
            widgets.HTML("<h3>ℹ️ Dataset Info</h3>"),
            self.info_output
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px', height='200px'))
        
        stats_box = widgets.VBox([
            widgets.HTML("<h3>📊 Statistics</h3>"),
            self.stats_output
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px', height='200px'))
        
        info_stats_row = widgets.HBox([info_box, stats_box])
        
        # Main layout
        main_layout = widgets.VBox([
            widgets.HTML(title_html),
            controls_row,
            plot_box,
            info_stats_row
        ])
        
        # Hide threshold slider initially if FDR is selected
        if self.correction_dropdown.value != 'none':
            self.threshold_slider.layout.display = 'none'
        
        return main_layout
    
    def display(self):
        """Display the interactive interface"""
        interface = self.create_interface()
        display(interface)
        
        # Initial plot if contrasts are available
        if self.contrasts:
            self.update_view(None)

# Convenience function
def launch_interactive_level2_viewer(level2_path):
    """Launch the interactive Level 2 viewer"""
    if not os.path.exists(level2_path):
        print(f"❌ Path not found: {level2_path}")
        return None
    
    print("🚀 Launching Interactive Level 2 fMRI Viewer...")
    print("=" * 50)
    
    viewer = InteractiveLevel2Viewer(level2_path)
    viewer.display()
    
    return viewer

# Quick setup function for Jupyter
def setup_level2_viewer():
    """Quick setup with instructions"""
    
    setup_html = """
    <div style="border: 2px solid #4CAF50; padding: 20px; border-radius: 10px; background-color: #f9f9f9;">
        <h2>🧠 Interactive Level 2 fMRI Viewer Setup</h2>
        
        <h3>📋 Prerequisites:</h3>
        <ul>
            <li><code>pip install ipywidgets nilearn scipy matplotlib nibabel</code></li>
            <li><code>jupyter nbextension enable --py widgetsnbextension</code></li>
        </ul>
        
        <h3>🚀 Quick Start:</h3>
        <pre><code># Replace with your Level 2 analysis path
level2_path = "/path/to/your/level2_analysis"

# Launch the viewer
viewer = launch_interactive_level2_viewer(level2_path)</code></pre>
        
        <h3>📁 Expected Directory Structure:</h3>
        <pre><code>level2_analysis/
├── contrast1_folder/
│   ├── SPM.mat
│   ├── con_0001.nii (or contrast.nii)
│   └── spmT_0001.nii (or spmT.nii)
├── contrast2_folder/
│   └── ...
└── ...</code></pre>
        
        <h3>🎯 Features:</h3>
        <ul>
            <li><strong>Real-time visualization</strong> with interactive controls</li>
            <li><strong>Multiple correction methods</strong>: FDR, Bonferroni, Uncorrected</li>
            <li><strong>Flexible display modes</strong>: Glass brain, orthogonal, mosaic</li>
            <li><strong>Statistical summaries</strong> with sample size information</li>
            <li><strong>Export functionality</strong> for thresholded maps</li>
            <li><strong>Comparison tools</strong> between correction methods</li>
        </ul>
        
        <h3>💡 Tips:</h3>
        <ul>
            <li>Start with <strong>FDR correction</strong> for most group analyses</li>
            <li>Use <strong>glass brain view</strong> for overview, <strong>orthogonal</strong> for precision</li>
            <li>Check <strong>sample sizes</strong> in the dataset info panel</li>
            <li><strong>Export maps</strong> for use in other software or publications</li>
        </ul>
    </div>
    """
    
    display(HTML(setup_html))

# Example analysis workflow
def demo_workflow():
    """Demonstrate typical analysis workflow"""
    
    workflow_html = """
    <div style="border: 2px solid #2196F3; padding: 20px; border-radius: 10px; background-color: #f0f8ff;">
        <h2>📊 Typical Level 2 Analysis Workflow</h2>
        
        <h3>1️⃣ Load and Explore</h3>
        <pre><code># Launch viewer
viewer = launch_interactive_level2_viewer("/path/to/level2")

# The interface will show:
# - Available contrasts with sample sizes
# - Dataset information
# - Interactive controls</code></pre>
        
        <h3>2️⃣ Initial Exploration</h3>
        <ul>
            <li>Select your contrast of interest</li>
            <li>Start with <strong>FDR correction (q < 0.05)</strong></li>
            <li>Use <strong>Glass brain view</strong> for overview</li>
            <li>Check the statistics panel for activation counts</li>
        </ul>
        
        <h3>3️⃣ Detailed Analysis</h3>
        <ul>
            <li>Click <strong>"Compare Corrections"</strong> to see all threshold methods</li>
            <li>Click <strong>"Threshold Summary"</strong> for detailed statistics</li>
            <li>Switch to <strong>orthogonal view</strong> for anatomical precision</li>
            <li>Try different alpha levels (0.001, 0.01, 0.05)</li>
        </ul>
        
        <h3>4️⃣ Publication Preparation</h3>
        <ul>
            <li>Use <strong>FDR q < 0.05</strong> for main results</li>
            <li>Export thresholded maps with <strong>"Export Maps"</strong></li>
            <li>Note peak coordinates and cluster sizes</li>
            <li>Consider effect sizes (contrast estimates)</li>
        </ul>
        
        <h3>5️⃣ Quality Checks</h3>
        <ul>
            <li>Verify sample sizes in the info panel</li>
            <li>Check for reasonable activation patterns</li>
            <li>Compare with uncorrected thresholds</li>
            <li>Look for potential artifacts or outliers</li>
        </ul>
    </div>
    """
    
    display(HTML(workflow_html))

# Statistical interpretation guide
def stats_guide():
    """Guide for interpreting statistical results"""
    
    stats_html = """
    <div style="border: 2px solid #FF9800; padding: 20px; border-radius: 10px; background-color: #fff8e1;">
        <h2>📈 Statistical Interpretation Guide</h2>
        
        <h3>🧮 Correction Methods</h3>
        <table style="border-collapse: collapse; width: 100%;">
            <tr style="background-color: #f2f2f2;">
                <th style="border: 1px solid #ddd; padding: 8px;">Method</th>
                <th style="border: 1px solid #ddd; padding: 8px;">Controls</th>
                <th style="border: 1px solid #ddd; padding: 8px;">Use Case</th>
                <th style="border: 1px solid #ddd; padding: 8px;">Typical Threshold</th>
            </tr>
            <tr>
                <td style="border: 1px solid #ddd; padding: 8px;"><strong>Uncorrected</strong></td>
                <td style="border: 1px solid #ddd; padding: 8px;">Nothing</td>
                <td style="border: 1px solid #ddd; padding: 8px;">Initial exploration only</td>
                <td style="border: 1px solid #ddd; padding: 8px;">t > 2.0 (p < 0.05)</td>
            </tr>
            <tr>
                <td style="border: 1px solid #ddd; padding: 8px;"><strong>FDR</strong></td>
                <td style="border: 1px solid #ddd; padding: 8px;">False discovery rate</td>
                <td style="border: 1px solid #ddd; padding: 8px;">Most publications</td>
                <td style="border: 1px solid #ddd; padding: 8px;">q < 0.05</td>
            </tr>
            <tr>
                <td style="border: 1px solid #ddd; padding: 8px;"><strong>Bonferroni</strong></td>
                <td style="border: 1px solid #ddd; padding: 8px;">Family-wise error rate</td>
                <td style="border: 1px solid #ddd; padding: 8px;">Confirmatory analysis</td>
                <td style="border: 1px solid #ddd; padding: 8px;">p < 0.05</td>
            </tr>
        </table>
        
        <h3>📊 Sample Size Guidelines</h3>
        <ul>
            <li><strong>N < 15:</strong> Very small - use conservative thresholds, report effect sizes</li>
            <li><strong>N = 15-25:</strong> Small to medium - FDR is appropriate</li>
            <li><strong>N > 25:</strong> Good power for group analysis</li>
            <li><strong>N > 50:</strong> High power - can detect smaller effects</li>
        </ul>
        
        <h3>🎯 Reporting Standards</h3>
        <ul>
            <li><strong>Always report:</strong> Correction method, threshold, sample size</li>
            <li><strong>Example:</strong> "FDR-corrected q < 0.05, N=24 subjects"</li>
            <li><strong>Include:</strong> Peak coordinates, cluster sizes, T-values</li>
            <li><strong>Consider:</strong> Effect size maps for biological interpretation</li>
        </ul>
        
        <h3>⚠️ Common Pitfalls</h3>
        <ul>
            <li><strong>Don't:</strong> Use uncorrected thresholds for final results</li>
            <li><strong>Don't:</strong> Cherry-pick threshold to get "significant" results</li>
            <li><strong>Don't:</strong> Ignore small sample size limitations</li>
            <li><strong>Do:</strong> Report null results when appropriate</li>
            <li><strong>Do:</strong> Consider cluster-extent thresholds</li>
        </ul>
    </div>
    """
    
    display(HTML(stats_html))

# All-in-one function for complete setup
def complete_level2_setup():
    """Complete setup with all guides"""
    setup_level2_viewer()
    demo_workflow()
    stats_guide()
    
    print("\n" + "="*50)
    print("🚀 Ready to start! Use:")
    print("viewer = launch_interactive_level2_viewer('/your/level2/path')")
    print("="*50)

if __name__ == "__main__":
    # Show setup instructions when run directly
    complete_level2_setup()

Method,Controls,Use Case,Typical Threshold
Uncorrected,Nothing,Initial exploration only,t > 2.0 (p < 0.05)
FDR,False discovery rate,Most publications,q < 0.05
Bonferroni,Family-wise error rate,Confirmatory analysis,p < 0.05



🚀 Ready to start! Use:
viewer = launch_interactive_level2_viewer('/your/level2/path')


In [5]:
viewer = launch_interactive_level2_viewer('/data00/projects/geoscan_v2/data/bids_data/derivatives_nocorrection/nipype/task-image_model-GEO-condition/l2analysis')

📊 ACTIVATION STATISTICS
Significant voxels: 29,333
Peak T-value: 14.095
Min T-value: -13.059
Brain coverage: 28.06%
